# Open-Weight LLM Deployment Pipeline — Kaggle E2E

Run this notebook in Kaggle with Internet enabled. A GPU accelerator is recommended but not mandatory for the API/unit-test stages. The default model is HuggingFaceTB/SmolLM2-360M. Model files are cached only in the Kaggle session and are not committed to Git.

In [ ]:
import os, sys, json, time, platform, subprocess, pathlib, statistics
from datetime import datetime, timezone

REPO_URL = "https://github.com/razz1992/OpenWeight-LLM-Deployment-Pipeline.git"
REPO_DIR = pathlib.Path("/kaggle/working/OpenWeight-LLM-Deployment-Pipeline")
MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Started UTC:", datetime.now(timezone.utc).isoformat())


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("Running CPU path; model inference may be slower.")


## 1. Install project dependencies

Kaggle images often include PyTorch already. This installs the project API/test/model libraries without reinstalling PyTorch.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")


## 2. API unit and validation tests

These tests do not require model weights or a GPU. They verify health response, unloaded-model handling and request validation.

In [ ]:
test_result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/unit"], capture_output=True, text=True)
print(test_result.stdout)
print(test_result.stderr)
assert test_result.returncode == 0, "Unit tests failed"


## 3. Model metadata and license review

Review the model card and license before using or redistributing weights. This cell retrieves the repository metadata; it does not grant redistribution rights.

In [ ]:
from huggingface_hub import HfApi
info = HfApi().model_info(MODEL_ID)
model_metadata = {"model_id": MODEL_ID, "revision": info.sha, "pipeline_tag": info.pipeline_tag,
                  "license": (info.card_data.license if info.card_data else None),
                  "private": info.private}
print(json.dumps(model_metadata, indent=2))
assert not info.private, "Expected a public model repository"


## 4. Load model and run inference

Use float16 on CUDA and float32 on CPU. The model is intentionally small to fit typical free Kaggle sessions.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch_dtype, device_map="auto" if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    model.to("cpu")
model.eval()
load_seconds = time.perf_counter() - t0
print("Loaded in seconds:", round(load_seconds, 3))


In [ ]:
def generate_text(prompt, max_new_tokens=96, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    kwargs = {"max_new_tokens": max_new_tokens, "do_sample": temperature > 0, "pad_token_id": tokenizer.eos_token_id}
    if temperature > 0:
        kwargs["temperature"] = temperature
    with torch.inference_mode():
        output = model.generate(**inputs, **kwargs)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

prompt = "Explain why monitoring latency matters when serving a language model:"
t0 = time.perf_counter()
answer = generate_text(prompt)
first_inference_seconds = time.perf_counter() - t0
print("Prompt:", prompt)
print("Output:", answer)
print("First inference seconds:", round(first_inference_seconds, 3))
assert isinstance(answer, str) and len(answer.strip()) > 0


## 5. Benchmark and resource telemetry

This is a small smoke benchmark, not a statistically rigorous performance study. Repeat with more prompts and warm-up iterations for production comparisons.

In [ ]:
prompts = ["What is an open-weight language model?", "Explain model quantization in simple terms.", "What does an inference API do?"]
latencies = []
for p in prompts:
    t0 = time.perf_counter(); out = generate_text(p, max_new_tokens=64); latencies.append(time.perf_counter()-t0)
    print({"prompt": p, "latency_seconds": round(latencies[-1], 3), "output_chars": len(out)})
try:
    import psutil
    ram_gib = psutil.virtual_memory().used / 1024**3
except Exception:
    ram_gib = None
if torch.cuda.is_available():
    gpu_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
else:
    gpu_peak_gib = None
benchmark = {"load_seconds": load_seconds, "first_inference_seconds": first_inference_seconds,
             "latency_seconds": latencies, "mean_latency_seconds": statistics.mean(latencies),
             "system_ram_used_gib": ram_gib, "gpu_peak_allocated_gib": gpu_peak_gib}
print(json.dumps(benchmark, indent=2))


## 6. Exercise the FastAPI endpoint with the loaded model

The API unit tests above validated the unloaded state. Here we inject the already-loaded notebook model into the API service and test a real generation request using FastAPI TestClient.

In [ ]:
from fastapi.testclient import TestClient
from api.main import app, model_service

# Adapt the notebook model to the service's generator interface.
def notebook_generator(prompt, **kwargs):
    generated = generate_text(prompt, max_new_tokens=kwargs.get("max_new_tokens", 96), temperature=kwargs.get("temperature", 0.0))
    return [{"generated_text": prompt + generated}]

model_service.generator = notebook_generator
client = TestClient(app)
health = client.get("/health")
api_response = client.post("/generate", json={"prompt": "Define inference in one sentence:", "max_new_tokens": 48, "temperature": 0.0})
print("Health:", health.status_code, health.json())
print("Generate status:", api_response.status_code)
print("Generate body:", api_response.json())
assert health.status_code == 200 and health.json()["model_loaded"] is True
assert api_response.status_code == 200
assert api_response.json()["generated_text"].strip()


## 7. Optional quantization capability check

Quantization is not forced because supported methods depend on GPU, CUDA, bitsandbytes and model architecture. This cell records whether 8-bit support appears available; it does not change the active model.

In [ ]:
quantization_check = {"cuda_available": torch.cuda.is_available()}
try:
    import bitsandbytes as bnb
    quantization_check["bitsandbytes_version"] = getattr(bnb, "__version__", "installed")
    quantization_check["8bit_candidate"] = bool(torch.cuda.is_available())
except Exception as exc:
    quantization_check["8bit_candidate"] = False
    quantization_check["reason"] = str(exc)[:300]
print(json.dumps(quantization_check, indent=2))


## 8. Save execution report

Download the JSON output from the Kaggle notebook's output panel, then add the measured values to docs/PROGRESS.md. Do not mark a stage complete until the notebook has actually run.

In [ ]:
report = {"project": "Open-Weight LLM Deployment Pipeline", "timestamp_utc": datetime.now(timezone.utc).isoformat(),
          "repository": REPO_URL, "commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
          "python": sys.version, "pytorch": torch.__version__, "cuda_available": torch.cuda.is_available(),
          "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
          "model": model_metadata, "benchmark": benchmark, "quantization_check": quantization_check,
          "api_health_status": health.status_code, "api_generate_status": api_response.status_code,
          "unit_test_return_code": test_result.returncode}
report_path = pathlib.Path("/kaggle/working/kaggle_pipeline_report.json")
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved:", report_path)
print(json.dumps(report, indent=2))


## 9. Pipeline boundaries

**Validated in Kaggle:** Python dependencies, API unit tests, model metadata lookup, model loading, inference, latency/resource smoke benchmark, API generation test and JSON report.

**Run outside Kaggle:** Docker image build and GitHub Actions CI should run on GitHub-hosted runners. Kaggle notebooks generally do not expose a privileged Docker daemon. AWS deployment is intentionally excluded; infrastructure templates can be validated without provisioning chargeable resources.